# Road Following - Live demo

In this notebook, we will use model we trained to move jetBot smoothly on track. 

### Load Trained Model

We will assume that you have already downloaded ``best_steering_model_xy.pth`` to work station as instructed in "train_model.ipynb" notebook. Now, you should upload model file to JetBot in to this notebook's directory. Once that's finished there should be a file named ``best_steering_model_xy.pth`` in this notebook's directory.

> Please make sure the file has uploaded fully before calling the next cell

Execute the code below to initialize the PyTorch model. This should look very familiar from the training notebook.

In [3]:
!pwd

/workspace/jetbot/notebooks/jacob_miner


In [1]:
MODEL_PATH = 'best_resnet18_224.pth'

In [2]:
import torchvision
import torch

model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
model.load_state_dict(torch.load(MODEL_PATH))
#model.eval()

<All keys matched successfully>

Next, load the trained weights from the ``best_steering_model_xy.pth`` file that you uploaded.

In [3]:
# model.load_state_dict(torch.load(MODEL_PATH))

Currently, the model weights are located on the CPU memory execute the code below to transfer to the GPU device.

In [4]:
device = torch.device('cuda')
model = model.to(device)
model = model.eval().half()

dummy = torch.zeros(1, 3, 224, 224).to(device).half()
with torch.no_grad():
    model = torch.jit.trace(model, dummy)
    for _ in range(20):
        model(dummy)

print('Model ready')

Model ready


### Creating the Pre-Processing Function

We have now loaded our model, but there's a slight issue. The format that we trained our model doesn't exactly match the format of the camera. To do that, we need to do some preprocessing. This involves the following steps:

1. Convert from HWC layout to CHW layout
2. Normalize using same parameters as we did during training (our camera provides values in [0, 255] range and training loaded images in [0, 1] range so we need to scale by 255.0
3. Transfer the data from CPU memory to GPU memory
4. Add a batch dimension

In [5]:
import torchvision.transforms as transforms
import torch.nn.functional as F
import cv2
import PIL.Image
import numpy as np

mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

Awesome! We've now defined our pre-processing function which can convert images from the camera format to the neural network input format.

Now, let's start and display our camera. You should be pretty familiar with this by now. 

In [6]:
from IPython.display import display
import ipywidgets
import traitlets
from jetbot import Camera, bgr8_to_jpeg

camera = Camera()

image_widget = ipywidgets.Image()

traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

We'll also create our robot instance which we'll need to drive the motors.

In [7]:
from jetbot import Robot

robot = Robot()

Now, we will define sliders to control JetBot
> Note: We have initialize the slider values for best known configurations, however these might not work for your dataset, therefore please increase or decrease the sliders according to your setup and environment

1. Speed Control (speed_gain_slider): To start your JetBot increase ``speed_gain_slider`` 
2. Steering Gain Control (steering_gain_slider): If you see JetBot is wobbling, you need to reduce ``steering_gain_slider`` till it is smooth
3. Steering Bias control (steering_bias_slider): If you see JetBot is biased towards extreme right or extreme left side of the track, you should control this slider till JetBot start following line or track in the center.  This accounts for motor biases as well as camera offsets

> Note: You should play around above mentioned sliders with lower speed to get smooth JetBot road following behavior.

In [8]:
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.1, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.0, description='steering bias')

display(speed_gain_slider, steering_gain_slider, steering_dgain_slider, steering_bias_slider)

FloatSlider(value=0.2, description='speed gain', max=1.0, step=0.01)

FloatSlider(value=0.1, description='steering gain', max=1.0, step=0.01)

FloatSlider(value=0.0, description='steering kd', max=0.5, step=0.001)

FloatSlider(value=0.0, description='steering bias', max=0.3, min=-0.3, step=0.01)



import ipywidgets as widgets
from IPython.display import display
import threading

running = widgets.ToggleButton(
    value=True,
    description='Running'
)

display(running)

def worker():
    while running.value:
        execute(camera.value)

threading.Thread(target=worker, daemon=True).start()Next, let's display some sliders that will let us see what JetBot is thinking.  The x and y sliders will display the predicted x, y values.

The steering slider will display our estimated steering value.  Please remember, this value isn't the actual angle of the target, but simply a value that is
nearly proportional.  When the actual angle is ``0``, this will be zero, and it will increase / decrease with the actual angle.  

In [9]:
x_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='y')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='speed')

display(ipywidgets.HBox([y_slider, speed_slider]))
display(x_slider, steering_slider)

FloatSlider(value=0.0, description='x', max=1.0, min=-1.0)

FloatSlider(value=0.0, description='steering', max=1.0, min=-1.0)

Next, we'll create a function that will get called whenever the camera's value changes. This function will do the following steps

1. Pre-process the camera image
2. Execute the neural network
3. Compute the approximate steering value
4. Control the motors using proportional / derivative control (PD)

In [10]:

mean_cpu = torch.tensor([0.485, 0.456, 0.406])
std_cpu  = torch.tensor([0.229, 0.224, 0.225])

def preprocess_cpu(image):
    # dokładnie jak live_demo, tylko bez cuda i half

    if isinstance(image, PIL.Image.Image):
        image = np.array(image)

    image = PIL.Image.fromarray(image)

    tensor = transforms.functional.to_tensor(image)

    tensor.sub_(mean_cpu[:, None, None])
    tensor.div_(std_cpu[:, None, None])

    return tensor.unsqueeze(0)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict_xy_cpu(image):
    tensor = preprocess_cpu(image)
    with torch.no_grad():
        xy = model(tensor).detach().cpu().numpy().flatten()
    return xy

mean_gpu = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std_gpu  = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

#   CHANGED PREPROCESSING TO AVOID PIL CONVERSION, AS CAMERA ALREADY GIVES US NUMPY HWC BGR =============================================================================
# def preprocess_gpu(image):
#     if isinstance(image, PIL.Image.Image):
#         image = np.array(image)
#     image = PIL.Image.fromarray(image)
#     tensor = transforms.functional.to_tensor(image).to(device).half()
#     tensor.sub_(mean_gpu[:, None, None]).div_(std_gpu[:, None, None])
#     return tensor.unsqueeze(0)

def preprocess_gpu(image):
    # image is already a numpy HWC BGR array from the camera
    tensor = torch.from_numpy(image).permute(2, 0, 1).float()  # HWC→CHW
    tensor = tensor[[2, 1, 0]]          # BGR→RGB
    tensor = tensor.to(device).half()
    tensor.div_(255.0)
    tensor.sub_(mean_gpu[:, None, None]).div_(std_gpu[:, None, None])
    return tensor.unsqueeze(0)

def predict_xy_gpu(image):
    tensor = preprocess_gpu(image)
    with torch.no_grad():
        xy = model(tensor).squeeze()
    return float(xy[0]), float(xy[1])

def compute_wheel_output(
    xy,
    speed_gain=0.2,
    steering_gain=0.2,
    steering_bias=0.0,
    steering_dgain=0.0,
    angle_last=0.0,
):
    x = xy[0] - 0.5
    y = 1.0 - xy[1]
    angle = np.arctan2(x, y)

    pid = angle * steering_gain + (angle - angle_last) * steering_dgain
    steering = pid + steering_bias

    # steering = speed_gain * steering_gain * angle + steering_bias
    
    left  = float(np.clip(speed_gain + steering, 0.0, 1.0))
    right = float(np.clip(speed_gain - steering, 0.0, 1.0))
    return {
        'x':       float(xy[0]),
        'y':       float(xy[1]),
        'angle':   float(angle),
        'steering': float(steering),
        'left':    left,
        'right':   right,
    }

In [11]:
# import threading
# import time
# execute_times = []

# executing = False
# execute_lock = threading.Lock()

# angle = 0.0
# angle_last = 0.0

# def execute(change):
#     now = time.time()
#     execute_times.append(now)
#     execute_times[:] = [t for t in execute_times if t > now - 1]
#     print(f'Executed {len(execute_times)} times in the last second', end='\r')
#     global angle, angle_last, executing

# #     with execute_lock:
# #         if executing:
# #             return
# #         executing = True

# #     try:
#     image = change['new']
#     pred_x, pred_y = predict_xy_gpu(image)

#     x = pred_x - 0.5
#     y = 1.0 - pred_y
#     angle = np.arctan2(x, y)

#     pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
#     angle_last = angle

#     steering_slider.value = pid + steering_bias_slider.value

#     left  = float(np.clip(speed_gain_slider.value + steering_slider.value, 0.0, 1.0))
#     right = float(np.clip(speed_gain_slider.value - steering_slider.value, 0.0, 1.0))

#     robot.left_motor.value  = left
#     robot.right_motor.value = right
        
# #     finally:
# #         with execute_lock:
# #             executing = False

# execute({'new': camera.value})

In [12]:
import matplotlib.pyplot as plt
import cv2
import time

# ALTERNATE STEERING LOGIC
angle = 0.0
angle_last = 0.0
current_steering = 0.0
ALPHA = 0.3
STRAIGHT_THRESHOLD = 0.05

def execute(image):
    global angle, angle_last, current_steering
    
#     plt.figure(figsize=(8, 8))
#     plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
#     plt.axis("off")
#     plt.show()

    xy = predict_xy_gpu(image)
    x  = xy[0] - 0.5
    y  = 1.0 - xy[1]
    angle = np.arctan2(0.5*x, y)

    pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
    angle_last = angle

    target_steering = 0.0 if abs(angle) < STRAIGHT_THRESHOLD else pid + steering_bias_slider.value

    current_steering = ALPHA * target_steering + (1 - ALPHA) * current_steering

    x_slider.value       = x
    y_slider.value       = y
    speed_slider.value   = speed_gain_slider.value
    steering_slider.value = current_steering

    left  = float(np.clip(speed_gain_slider.value + current_steering, 0.0, 1.0))
    right = float(np.clip(speed_gain_slider.value - current_steering, 0.0, 1.0))

    robot.left_motor.value  = left
    robot.right_motor.value = right

# execute(camera.value)

Cool! We've created our neural network execution function, but now we need to attach it to the camera for processing.

We accomplish that with the observe function.

>WARNING: This code will move the robot!! Please make sure your robot has clearance and it is on Lego or Track you have collected data on. The road follower should work, but the neural network is only as good as the data it's trained on!

In [5]:
import ipywidgets as widgets
from IPython.display import display
import threading
from time import time

running = widgets.ToggleButton(
    value=True,
    description='Stop'
)

display(running)

def worker():
    while running.value:
        execute(camera.value)

threading.Thread(target=worker, daemon=True).start()

ToggleButton(value=True, description='Running')

1781015516.8578786

In [1]:
robot.stop()

NameError: name 'robot' is not defined

Awesome! If your robot is plugged in it should now be generating new commands with each new camera frame. 

You can now place JetBot on  Lego or Track you have collected data on and see whether it can follow track.

If you want to stop this behavior, you can unattach this callback by executing the code below.

In [ ]:
# camera.observe(execute, names='value')

In [81]:
# import time

# robot.stop()

# camera.unobserve(execute, names='value')

# time.sleep(0.1)  # add a small sleep to make sure frames have finished processing

# robot.stop()

Again, let's close the camera conneciton properly so that we can use the camera in other notebooks.

In [ ]:
camera.stop()

### Conclusion
That's it for this live demo! Hopefully you had some fun seeing your JetBot moving smoothly on track following the road!!!

If your JetBot wasn't following road very well, try to spot where it fails. The beauty is that we can collect more data for these failure scenarios and the JetBot should get even better :)